# Enrich Court Authority Cards — Qwen3.5-35B-A3B

Enriches `court_authority_cards_v4.jsonl` with LLM-derived semantic fields for RAG.

**Model:** `Qwen/Qwen3.5-35B-A3B` (MoE, 35B total / 3B active params)  
**Why:** Ranks #3 globally on the Structured Output Benchmark (SOB, arxiv 2604.25359) with Value Accuracy 0.801 — best open-weight model for JSON schema compliance.  
**GPU target:** NVIDIA G4 / RTX PRO 6000 Blackwell (96 GB VRAM)  
**Thinking mode:** DISABLED — required for guided JSON decoding (thinking tokens break schema constraints).

---
Adapted from `scripts/enrich_cards_with_qwen.py`.

## 1 · Environment setup

In [2]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    # Verify GPU
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())

Running in Colab: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [2]:
if IN_COLAB:
    # vLLM nightly recommended for Qwen3.5-35B-A3B MoE support
    %pip install -q --upgrade --pre vllm
    %pip install -q tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/53

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Configuration

In [4]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR    = BASE_DIR / 'data'
INSIGHTS_DIR= BASE_DIR / 'data_insights'
ART_DIR     = BASE_DIR / 'artifacts'
SCRIPT_DIR  = BASE_DIR / 'scripts'

for d in [DATA_DIR, INSIGHTS_DIR, ART_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
# Qwen3.5-35B-A3B: MoE, 35B total / 3B active, ranks #3 on SOB structured output benchmark.
# Thinking mode is forcibly DISABLED — required for guided JSON decoding.
#
# AWQ variant (saves ~52 GB VRAM, recommended if available on your HuggingFace mirror):
#   MODEL_ID = 'bartowski/Qwen3.5-35B-A3B-AWQ'   # community AWQ — check HF for latest
#   QUANTIZATION = 'awq'
#
# Default: official BF16 — needs ~70 GB weights + KV cache; fits on 96 GB.
MODEL_ID     = 'Qwen/Qwen3.5-35B-A3B'
QUANTIZATION = None     # None for BF16; 'awq' if using a pre-quantized AWQ variant

# ── Inference ────────────────────────────────────────────────────────────────
GPU_MEMORY_UTIL    = 0.87   # 0.87 × 96 GB ≈ 83 GB — leaves headroom for KV cache
MAX_MODEL_LEN      = 4096
BATCH_SIZE         = 64     # MoE is faster per token; bump up vs dense 32B
TEMPERATURE        = 0.15   # lower = more deterministic JSON
MAX_TOKENS         = 600
TENSOR_PARALLEL    = 1      # single GPU; set 2+ if multi-GPU

# ── Files ────────────────────────────────────────────────────────────────────
INPUT_FILE      = ART_DIR / 'court_authority_cards_v4.jsonl'
OUTPUT_FILE     = ART_DIR  / 'court_authority_cards_rag.jsonl'
CHECKPOINT_FILE = ART_DIR  / 'rag_checkpoint.txt'

# Optional: stop after N cards (0 = process all)
LIMIT = 1000

print('BASE_DIR   :', BASE_DIR)
print('INPUT_FILE :', INPUT_FILE)
print('OUTPUT_FILE:', OUTPUT_FILE)
print('MODEL_ID   :', MODEL_ID)

BASE_DIR   : /content/drive/MyDrive/swiss_law
INPUT_FILE : /content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_v4.jsonl
OUTPUT_FILE: /content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_rag.jsonl
MODEL_ID   : Qwen/Qwen3.5-35B-A3B


## 3 · JSON schema + prompts

In [5]:
import re

# ── Pre-filter: trivial paragraphs that don't need an LLM ────────────────────
COST_PROC_RE = re.compile(
    r'(?:'
    r'\bgerichtskosten\b|\bprozesskosten\b|\bverfahrenskosten\b|'
    r'\bfrais judiciaires\b|\bfrais de la cause\b|\bd[eé]pens\b|'
    r'\bspese giudiziarie\b|\bripetibili\b|'
    r'\bparteientsch[äa]digung\b|\bhonoraire\b|'
    r'\bunentgeltliche rechtspflege\b|\bassistance judiciaire\b|'
    r'\bpatrocinio gratuito\b|'
    r'\bdie sache wird .{0,80}zur[üu]ckgewiesen\b|'
    r'\brenvoyer la cause\b|'
    r'\bla causa [eè] rinviata\b|'
    r'^\s*\d+\.\s*\d+\..{0,5}fr\.\s*\d'
    r')',
    re.IGNORECASE | re.MULTILINE,
)

# ── JSON schema for guided generation ────────────────────────────────────────
RAG_SCHEMA = {
    'type': 'object',
    'properties': {
        'english_summary':          {'type': 'string'},
        'legal_topic':              {'type': 'string'},
        'legal_question':           {'type': 'string'},
        'legal_rule':               {'type': 'string'},
        'court_holding':            {'type': 'string'},
        'factual_context':          {'type': 'string'},
        'english_legal_concepts':   {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
        'search_keywords':          {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
        'natural_language_queries': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
        'paragraph_role': {
            'type': 'string',
            'enum': ['holding', 'reasoning', 'background', 'cost',
                     'procedural', 'disposition', 'standard_of_review', 'obiter'],
        },
        'outcome_signal': {
            'type': 'string',
            'enum': ['granted', 'dismissed', 'inadmissible', 'remitted', 'partial', 'none'],
        },
    },
    'required': [
        'english_summary', 'legal_topic',
        'english_legal_concepts', 'search_keywords',
        'natural_language_queries', 'paragraph_role', 'outcome_signal',
    ],
}

SYSTEM_PROMPT = (
    'You are a Swiss legal analyst. The user gives you a paragraph from a Swiss '
    'Federal Tribunal decision in German, French, or Italian. '
    'Translate every concept into precise English legal terminology and emit '
    'structured JSON to power English-language semantic-search RAG. '
    'Be concrete: prefer \'extension of pretrial detention based on flight risk\' '
    'over \'detention\'. Output ONLY the JSON object, no preamble.'
)

print('Schema fields:', list(RAG_SCHEMA['properties'].keys()))

Schema fields: ['english_summary', 'legal_topic', 'legal_question', 'legal_rule', 'court_holding', 'factual_context', 'english_legal_concepts', 'search_keywords', 'natural_language_queries', 'paragraph_role', 'outcome_signal']


## 4 · Helper functions

In [6]:
import json
from typing import Iterator


def build_user_message(card: dict) -> str:
    text       = card.get('text_excerpt_original', '')[:2500]
    citation   = card.get('citation', '')
    legal_area = card.get('legal_area', '')
    existing   = card.get('issue_labels_en') or []
    parts = [
        f'Citation: {citation}',
        f'Legal area (deterministic): {legal_area}',
    ]
    if existing:
        labels = ', '.join(existing[:8])
        parts.append(f'Existing labels: {labels}')
    parts += ['', 'Paragraph (original language):', text, '', 'Produce the JSON now.']
    return '\n'.join(parts)


def _stub(summary, topic, concepts, keywords, role, method):
    return {
        'english_summary':          summary,
        'legal_topic':              topic,
        'legal_question':           '',
        'legal_rule':               '',
        'court_holding':            '',
        'factual_context':          '',
        'english_legal_concepts':   concepts,
        'search_keywords':          keywords,
        'natural_language_queries': [],
        'paragraph_role':           role,
        'outcome_signal':           'none',
        'method':                   method,
    }


def auto_classify(card: dict) -> dict | None:
    if card.get('is_notification_paragraph'):
        return _stub(
            'Procedural notification of the judgment to the parties.',
            'judgment notification',
            ['service of judgment'],
            ['notification', 'service', 'judgment communication'],
            role='procedural', method='auto_notification',
        )
    text = card.get('text_excerpt_original', '') or ''
    if len(text) < 50:
        return _stub(
            'Short procedural fragment (cross-reference or one-line ruling).',
            'procedural fragment',
            [], [], role='procedural', method='auto_short',
        )
    if COST_PROC_RE.search(text[:400]):
        return _stub(
            'Court-cost or procedural-fee allocation paragraph.',
            'court costs and procedural fees',
            ['court costs', 'procedural fees', 'legal aid'],
            ['costs', 'court fees', 'frais judiciaires', 'Gerichtskosten'],
            role='cost', method='auto_cost',
        )
    return None


def stream_input(path: Path, start_offset: int) -> Iterator[tuple[int, dict]]:
    with path.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < start_offset or not line.strip():
                continue
            try:
                yield i, json.loads(line)
            except json.JSONDecodeError:
                continue


def count_lines(path: Path) -> int:
    n = 0
    with path.open('rb') as f:
        for _ in f:
            n += 1
    return n


print('Helpers loaded.')

Helpers loaded.


## 5 · Load model

> **Note:** Loading Qwen3.5-35B-A3B in BF16 downloads ~70 GB from HuggingFace Hub on first run.  
> Subsequent runs load from the Colab disk cache (typically `/root/.cache/huggingface`).
>
> **Thinking mode is explicitly disabled** via `chat_template_kwargs={"enable_thinking": False}`.  
> Without this, the model emits `<think>…</think>` tokens that break guided JSON decoding.

In [11]:
!pip install -U transformers accelerate

In [17]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 97.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


In [3]:
!pip install --upgrade vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 199.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 173.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 142.7

In [ ]:
import os
from vllm import LLM, SamplingParams
# If vLLM throws a type error on speculative_config, you may need to import it:
# from vllm.config import SpeculativeConfig

# Force a clean multiprocessing start to prevent Jupyter silent crashes
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

print("Loading Qwen3.5 via vLLM with Latency-Focused Serving...")

llm = LLM(
    model="Qwen/Qwen3.5-35B-A3B-Base",
    quantization=None,
    dtype='bfloat16',
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    trust_remote_code=False,
    tensor_parallel_size=1,
    enforce_eager=True,
    limit_mm_per_prompt={"image": 0, "video": 0}, # Keeps the multimodal encoder disabled

    # -----------------------------------------------------
    # LATENCY-FOCUSED CONFIGURATION (From vLLM Docs)
    # -----------------------------------------------------

    # 1. Disable prefix caching (Required for MTP efficiency)
    enable_prefix_caching=False,

    # 2. Enable MTP-1 Speculative Decoding
    speculative_config={"method": "mtp", "num_speculative_tokens": 1},

    # 3. Add the native Qwen3 reasoning parser
    reasoning_parser="qwen3",
)

# Set up the sampling parameters WITH the guided JSON schema
sampling_params = SamplingParams(
    temperature=0.15,
    max_tokens=600,
    guided_decoding={"json": RAG_SCHEMA} # Forces perfect JSON output
)

# Disable the thinking tokens as requested by your script setup
NO_THINK = {"enable_thinking": False}

print('Model loaded successfully for minimum latency!')

Loading Qwen3.5 via vLLM with Latency-Focused Serving...
INFO 04-30 17:43:50 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'reasoning_parser': 'qwen3', 'speculative_config': {'method': 'mtp', 'num_speculative_tokens': 1}, 'model': 'Qwen/Qwen3.5-35B-A3B-Base'}


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

INFO 04-30 17:44:02 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 04-30 17:44:02 [nixl_utils.py:34] NIXL is not available
WARNING 04-30 17:44:02 [nixl_utils.py:44] NIXL agent config is not available
INFO 04-30 17:44:02 [model.py:555] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 04-30 17:44:02 [model.py:1680] Using max model len 4096
INFO 04-30 17:44:10 [model.py:555] Resolved architecture: Qwen3_5MoeMTP
INFO 04-30 17:44:10 [model.py:1680] Using max model len 262144
INFO 04-30 17:44:10 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-30 17:44:10 [vllm.py:840] Asynchronous scheduling is enabled.
WARNING 04-30 17:44:10 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-30 17:44:10 [vllm.py:914] Inductor compilation was disabled by user settings, optimizatio

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

INFO 04-30 17:44:13 [compilation.py:303] Enabled custom fusions: norm_quant, act_quant


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 04-30 17:44:13 [registry.py:126] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.


## 6 · Run enrichment

In [16]:
from tqdm.auto import tqdm
import json

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input not found: {INPUT_FILE}')

# 👈 FIX 1: Fix the Hugging Face batched generation warning
tokenizer.padding_side = "left"

start = 0
if CHECKPOINT_FILE.exists():
    try:
        start = int(CHECKPOINT_FILE.read_text().strip() or '0')
    except ValueError:
        start = 0
print(f'Resuming at line {start:,}')

total = count_lines(INPUT_FILE)
if LIMIT:
    total = min(total, start + LIMIT)
print(f'Total={total:,}  To process={total - start:,}')

out_f = OUTPUT_FILE.open('a', encoding='utf-8')
pbar = tqdm(total=total, initial=start, desc='enrich', unit='card', smoothing=0.05)
pending = []
json_errors = 0

def flush_batch():
    global pending, json_errors
    if not pending:
        return

    # 1. Prepare prompts using the chat template
    texts = []
    for _, card in pending:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_message(card)}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        texts.append(text)

    # 2. Tokenize and pad the batch (now safely left-padded!)
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=3000).to(model.device)

    # 3. Generate outputs
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 4. Decode and parse JSON
    input_lengths = inputs.input_ids.shape[1]
    for (line_idx, card), output in zip(pending, outputs):
        raw = tokenizer.decode(output[input_lengths:], skip_special_tokens=True).strip()

        # Strip markdown formatting if Qwen added it
        if raw.startswith("```json"):
            raw = raw[7:]
        if raw.startswith("```"):
            raw = raw[3:]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()

        try:
            enriched = json.loads(raw)

            # 👈 FIX 2: Ensure the model actually returned a dictionary!
            if not isinstance(enriched, dict):
                raise ValueError(f"Model output a {type(enriched).__name__} instead of a dict.")

            enriched['method'] = 'qwen35_35b_a3b_hf'

        except (json.JSONDecodeError, ValueError) as e:
            json_errors += 1
            enriched = _stub('', '', [], [], role='reasoning', method='json_parse_failed')
            enriched['raw_output'] = raw[:400]

        card['rag_enrichment'] = enriched
        out_f.write(json.dumps(card, ensure_ascii=False) + '\n')

    out_f.flush()
    CHECKPOINT_FILE.write_text(str(pending[-1][0] + 1))
    pbar.update(len(pending))
    pending.clear()

# --- Main Loop ---
processed = 0
for line_idx, card in stream_input(INPUT_FILE, start):
    if LIMIT and processed >= LIMIT:
        break

    auto = auto_classify(card)
    if auto is not None:
        card['rag_enrichment'] = auto
        out_f.write(json.dumps(card, ensure_ascii=False) + '\n')
        CHECKPOINT_FILE.write_text(str(line_idx + 1))
        pbar.update(1)
        processed += 1
        continue

    pending.append((line_idx, card))
    if len(pending) >= BATCH_SIZE:
        flush_batch()
    processed += 1

flush_batch()
out_f.close()
pbar.close()

print(f'Done. JSON parse errors: {json_errors}')
print(f'Output → {OUTPUT_FILE}')

Resuming at line 24
Total=1,024  To process=1,000


enrich:   2%|2         | 24/1024 [00:00<?, ?card/s]

KeyboardInterrupt: 

## 7 · Verify output

In [ ]:
from collections import Counter

roles    = Counter()
outcomes = Counter()
methods  = Counter()
total_out = 0
missing_required = 0

REQUIRED = RAG_SCHEMA['required']

with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        e    = card.get('rag_enrichment', {})
        total_out += 1
        roles[e.get('paragraph_role', 'MISSING')]    += 1
        outcomes[e.get('outcome_signal', 'MISSING')] += 1
        methods[e.get('method', 'MISSING')]          += 1
        if any(k not in e for k in REQUIRED):
            missing_required += 1

print(f'Total output cards : {total_out:,}')
print(f'Missing required   : {missing_required}')
print()
print('paragraph_role distribution:')
for k, v in roles.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('outcome_signal distribution:')
for k, v in outcomes.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('method distribution:')
for k, v in methods.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')

In [ ]:
# Show 3 random LLM-enriched cards for a quick quality check
import random

llm_cards = []
with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        if card.get('rag_enrichment', {}).get('method', '').startswith('qwen35'):
            llm_cards.append(card)

for card in random.sample(llm_cards, min(3, len(llm_cards))):
    e = card['rag_enrichment']
    print('─' * 70)
    print('Citation     :', card.get('citation', ''))
    print('Role         :', e.get('paragraph_role'))
    print('Outcome      :', e.get('outcome_signal'))
    print('Topic        :', e.get('legal_topic'))
    print('Summary      :', e.get('english_summary', '')[:200])
    print('Keywords     :', e.get('search_keywords'))
    print('NL queries   :', e.get('natural_language_queries'))
    print()